In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


llm = ChatOllama(
    model="qwen2.5:7b",       
    temperature=0.7,
    base_url="http://localhost:11434"  
)

d:\year 3 materials\year 4\recommendation system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
parser = StrOutputParser()
prompt_template = ChatPromptTemplate.from_template("{prompt}")

In [3]:
chain =  prompt_template | llm |parser
response = chain.invoke({"prompt":"give me the ingredients to bake a cake"})
response

"Sure! Here's a basic recipe for a vanilla sponge cake:\n\n### Ingredients:\n- 175 grams (about 3 and a half cups) all-purpose flour\n- 170 grams unsalted butter, softened\n- 200 grams caster sugar\n- 4 large eggs, at room temperature\n- 1 tsp baking powder\n- 1 tsp vanilla extract\n\n### Optional Toppings:\n- Icing sugar (for dusting)\n- Fresh fruits or chocolate shavings (for decoration)\n\n### Instructions:\n\n1. **Preheat Oven and Prepare Cake Tin:**\n   - Preheat your oven to 350°F (175°C).\n   - Grease a 9-inch round cake tin with butter or line it with parchment paper.\n\n2. **Cream Butter and Sugar:**\n   - Using an electric mixer, beat the softened butter and sugar together until light and fluffy. This usually takes about 3-4 minutes.\n\n3. **Add Eggs:**\n   - Crack in the eggs one at a time, mixing well after each addition. Ensure that each egg is fully incorporated before adding the next one.\n\n4. **Mix Dry Ingredients:**\n   - In a separate bowl, sift together the flour an

In [4]:
import pandas as pd

df = pd.read_csv("phone_cleaned.csv")

# Pull real, canonical values from the dataset
available_brands = sorted(df["brand"].dropna().unique().tolist())
available_categories = sorted(df["category_l3"].dropna().unique().tolist())
available_storage = sorted(df["storage_gb"].dropna().unique().tolist())
available_ram = sorted(df["ram_gb"].dropna().unique().tolist())
available_networks = sorted(df["network"].dropna().unique().tolist())

price_min, price_max = df["price_egp"].min(), df["price_egp"].max()

print(available_brands)
print(available_categories)
print(available_storage, available_ram, available_networks)
print(price_min, price_max)

['Apple', 'Digit', 'HMD', 'Honor', 'Hope', 'Huawei', 'Infinix', 'Itel', 'Motorola', 'Nokia', 'Nothing', 'Oppo', 'Realme', 'Samsung', 'Techno', 'Tecno', 'Vivo', 'Xiaomi']
['Feature Phones', 'Re-furbished Mobiles', 'Smart Phones']
[8.0, 32.0, 64.0, 128.0, 256.0, 265.0, 512.0, 1024.0, 2048.0] [2.0, 3.0, 4.0, 6.0, 8.0, 12.0, 16.0, 128.0] ['4G', '4G LTE', '5G', 'LTE']
475.0 189999.0


In [5]:
SLOTS = {
    "budget_egp": None,
    "use_case": None,       # gaming, camera, business, general
    "storage_gb": None,
    "ram_gb": None,
    "brand_pref": None,
    "network": None,        # 4G/5G
}

SYSTEM_PROMPT_TEMPLATE = """You are a phone-buying assistant for the Egyptian market.

You must collect these preferences from the user through natural conversation, 
one or two questions at a time:

- budget_egp: a numeric value or range (dataset range: {price_min} - {price_max} EGP)
- use_case: must map to one of these categories: {categories}
- storage_gb: must be one of: {storage_options}
- ram_gb: must be one of: {ram_options}
- brand_pref: must be one of these brands, or "no preference": {brands}
- network: must be one of: {networks}

Rules:
- Ask ONE clarifying question at a time.
- If the user gives a value that doesn't match the allowed options, gently 
  map it to the closest valid option, or ask them to choose from the list.
- Do NOT recommend or mention specific phone models yet.
- Once ALL slots are filled, respond with a JSON object summarizing the 
  preferences, wrapped in <PREFS>...</PREFS> tags, and nothing else.
- Keep questions short and conversational.
"""

system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
    price_min=int(price_min),
    price_max=int(price_max),
    categories=", ".join(available_categories),
    storage_options=", ".join(str(x) for x in available_storage),
    ram_options=", ".join(str(x) for x in available_ram),
    brands=", ".join(available_brands),
    networks=", ".join(available_networks),
)

In [6]:

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])

from difflib import get_close_matches

def validate_prefs(prefs: dict) -> dict:
    validated = dict(prefs)

    if prefs.get("brand_pref") and prefs["brand_pref"].lower() != "no preference":
        match = get_close_matches(prefs["brand_pref"], available_brands, n=1, cutoff=0.6)
        validated["brand_pref"] = match[0] if match else None

    if prefs.get("use_case"):
        match = get_close_matches(prefs["use_case"], available_categories, n=1, cutoff=0.5)
        validated["use_case"] = match[0] if match else prefs["use_case"]

    if prefs.get("storage_gb"):
        closest = min(available_storage, key=lambda x: abs(x - float(prefs["storage_gb"])))
        validated["storage_gb"] = closest

    if prefs.get("ram_gb"):
        closest = min(available_ram, key=lambda x: abs(x - float(prefs["ram_gb"])))
        validated["ram_gb"] = closest

    return validated


In [7]:
import json, re

class PhoneChatbot:
    def __init__(self, llm, system_prompt: str):
        self.chat_template = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("placeholder", "{chat_history}"),
        ])
        self.chain = self.chat_template | llm | StrOutputParser()
        self.chat_history = []
        self.prefs = None  # will hold extracted preferences once complete

    def ask(self, user_input: str) -> str:
        self.chat_history.append(("human", user_input))
        response = self.chain.invoke({"chat_history": self.chat_history})
        self.chat_history.append(("ai", response))

        extracted = self._extract_prefs(response)
        if extracted:
            self.prefs = extracted

        return response

    def _extract_prefs(self, response_text: str):
        match = re.search(r"<PREFS>\s*(.*?)\s*</PREFS>", response_text, re.DOTALL)
        if not match:
            return None
        raw = match.group(1).strip()
        raw = re.sub(r"^```json|```$", "", raw, flags=re.MULTILINE).strip()
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return None

    def is_done(self) -> bool:
        return self.prefs is not None

In [8]:
bot = PhoneChatbot(llm, system_prompt=system_prompt)  # system_prompt built from your dataset values

print("AI: Hi! I can help you find a phone. What's your budget?")

while not bot.is_done():
    user_input = input("You: ")
    if user_input.lower() in ("quit", "exit"):
        break
    response = bot.ask(user_input)
    print("AI:", response)

if bot.is_done():
    print("\n✅ Preferences collected:")
    print(bot.prefs)

AI: Hi! I can help you find a phone. What's your budget?
AI: Great! Let's narrow that down. Are you looking for a Smart Phone or a Feature Phone within your budget?
AI: Got it! For a good camera experience, we typically recommend Smart Phones. Would you prefer a Smart Phone within your 7000 EGP budget?
AI: Perfect! Now, let's consider storage. Which of these options sounds right for you: 8 GB, 32 GB, 64 GB, 128 GB, or do you have a preference?
AI: Alright, for good camera performance and staying within your budget, we might look at options with more storage. How about we consider 32 GB or 64 GB of storage? Which one do you prefer?
AI: Great choice! Now, let's think about the RAM. Would you like 4 GB, 8 GB, or 12 GB of RAM for your phone?
AI: Within your budget, we can usually get up to 6 GB or 8 GB of RAM. Which one do you prefer: 4 GB, 6 GB, or 8 GB?
AI: Perfect! Now, let's decide on the brand. Do you have any preference among these brands: Apple, Digit, HMD, Honor, Hope, Huawei, Infi

In [9]:
def filter_candidates(df, prefs, tolerance=0.15, top_n=15):
    filtered = df[df["in_stock"] == True].copy()
    
    if prefs.get("budget_egp"):
        budget = float(prefs["budget_egp"])
        filtered = filtered[
            (filtered["price_egp"] >= budget * (1 - tolerance)) &
            (filtered["price_egp"] <= budget * (1 + tolerance))
        ]
    
    if prefs.get("storage_gb"):
        filtered = filtered[filtered["storage_gb"] >= prefs["storage_gb"]]
    
    if prefs.get("ram_gb"):
        filtered = filtered[filtered["ram_gb"] >= prefs["ram_gb"]]
    
    if prefs.get("brand_pref") and prefs["brand_pref"].lower() != "no preference":
        filtered = filtered[filtered["brand"].str.lower() == prefs["brand_pref"].lower()]
    
    if prefs.get("network"):
        filtered = filtered[filtered["network"].str.contains(prefs["network"], case=False, na=False)]
    
    return filtered.sort_values("price_egp").head(top_n)

In [10]:
if bot.is_done():
    validated_prefs = validate_prefs(bot.prefs)  # from the fuzzy-matching step earlier
    candidates = filter_candidates(df, validated_prefs)
    print(candidates[["brand", "model", "price_egp", "storage_gb", "ram_gb"]])

Empty DataFrame
Columns: [brand, model, price_egp, storage_gb, ram_gb]
Index: []
